# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis.** In the warehouse, one row = one report date × one client × one content item (`fact_content_daily_performance`, partition `month=2026-03`). Section 3 checks this: no duplicate keys were found.

**What my model rows are.** One row = one content item (within one client), summarised over two halves of March: 1–15 March (used for features) and 16–31 March (used only to build the label).

**Time window.** 2026-03-01 to 2026-03-31, a mid-panel month. The final month (June 2026) is a sealed test month and I do not touch it.

**Slice.** Only rows where `gsc_data_available IS TRUE` (my lane is refresh / content opportunity scoring, which relies on Search Console data).

**What I rank.** Content items by refresh priority. Proxy label: "decline" = clicks in 16–31 March are below 80% of clicks in 1–15 March, among items with at least 5 clicks in the first half. This is a directional, decision-support proxy, not a measure of true content quality.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.execute(f"CREATE VIEW mar AS SELECT * FROM read_parquet('{MAR}')")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (all computed from 1–15 March only, so knowable at the decision moment):
`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, plus `report_date` used only to count active days.

**Label:** `gsc_clicks` from 16–31 March, used only to build the decline label. It is never a feature.

**Context** (used to group or filter, not as features): `client_hash_id` (client-holdout split), `content_hash_id`, `report_date`, `gsc_data_available`, `month`.

**Excluded, with why:**
- All `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` columns: they belong to other lanes, and GA4 is available on only about 4.2% of March rows (Query 3).
- Any column from 16–31 March as a feature: it would be future information.
- June 2026 (final month): sealed test month, not used for developing label logic.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
buckets = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position (1-15 March only)"],
    "label": ["gsc_clicks (16-31 March only, used to build the decline label)"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "gsc_data_available", "month"],
    "excluded": ["ga4_*", "sessions_*", "ai_*", "scroll_events", "anything from 16-31 March as a feature", "June 2026"],
}
for k, v in buckets.items():
    print(f"{k:9} -> {v}")


feature   -> ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position (1-15 March only)']
label     -> ['gsc_clicks (16-31 March only, used to build the decline label)']
context   -> ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_data_available', 'month']
excluded  -> ['ga4_*', 'sessions_*', 'ai_*', 'scroll_events', 'anything from 16-31 March as a feature', 'June 2026']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Q1 grain: duplicate (report_date, client, content) keys. 0 rows = grain holds")
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM mar
GROUP BY ALL
HAVING COUNT(*) > 1
LIMIT 10
""").show()

print("Q2 row count and date span")
con.sql("""
SELECT COUNT(*) AS n_rows,
       MIN(report_date) AS first_day,
       MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items
FROM mar
""").show()

print("Q3 availability (IS TRUE)")
avail = con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM mar
""").df()
print(avail.to_string(index=False))

Q1 grain: duplicate (report_date, client, content) keys. 0 rows = grain holds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

Q2 row count and date span
┌─────────┬────────────┬────────────┬─────────┬───────────────┐
│ n_rows  │ first_day  │  last_day  │ clients │ content_items │
│  int64  │    date    │    date    │  int64  │     int64     │
├─────────┼────────────┼────────────┼─────────┼───────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │      55 │        331437 │
└─────────┴────────────┴────────────┴─────────┴───────────────┘

Q3 availability (IS TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available  ga4_available
    9841378        3611061         413966


**Observed in March 2026 (measured with the three queries above):**
- Q1: no duplicate (report_date, client, content) keys, so the stated grain holds.
- Q2: 9,841,378 rows from 2026-03-01 to 2026-03-31, covering 55 clients and 331,437 content items.
- Q3: only 3,611,061 rows (about 36.7%) have `gsc_data_available IS TRUE`, and 413,966 rows (about 4.2%) have `ga4_data_available IS TRUE`. My slice is therefore the 3,611,061 Search Console rows, not all of March.

In [4]:
import numpy as np

frame = con.sql("""
SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS avg_position_h1,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS active_days_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '2026-03-15') AS clicks_h2
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df = frame.copy()
df["clicks_h2"] = df["clicks_h2"].fillna(0)
df["ctr_h1"] = df["clicks_h1"] / df["impressions_h1"].replace(0, np.nan)
FEATURES = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
df = df[df["clicks_h1"] >= 5].dropna(subset=FEATURES)
df["label"] = (df["clicks_h2"] < 0.8 * df["clicks_h1"]).astype(int)
df = df.sample(min(len(df), 200_000), random_state=0)

print("rows:", len(df), "| decline rate:", round(df["label"].mean(), 3))
df[FEATURES + ["label"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 17431 | decline rate: 0.442


,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,label
154042,1656.0,7.0,0.004227,21.248086,15,1
149357,340.0,6.0,0.017647,22.342052,10,0
6264,13351.0,113.0,0.008464,11.062513,15,0
3487,1166.0,6.0,0.005146,26.639829,15,0
137514,7981.0,22.0,0.002757,6.195841,15,0


**Five features, each knowable at the decision moment (end of 15 March):**
- `impressions_h1`: total Search impressions from 1–15 March only, so it is known by the decision date.
- `clicks_h1`: total clicks from 1–15 March only, recorded before the decision date.
- `ctr_h1`: clicks_h1 divided by impressions_h1, both from the first half.
- `avg_position_h1`: mean of daily average Google position over 1–15 March.
- `active_days_h1`: number of days in 1–15 March with at least one impression.

None of them uses any row dated 16 March or later. The label comes from 16–31 March only.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    tr, te = next(GroupShuffleSplit(test_size=0.25, random_state=0)
                  .split(df, groups=df["client_hash_id"]))
    m = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1, random_state=0)
    m.fit(df.iloc[tr][cols], df.iloc[tr]["label"])
    return roc_auc_score(df.iloc[te]["label"], m.predict_proba(df.iloc[te][cols])[:, 1])

print("1) honest   :", round(quick_score(FEATURES), 3))

df["leak_clicks_h2"] = df["clicks_h2"]            # label-derived column, added on purpose
print("2) with leak:", round(quick_score(FEATURES + ["leak_clicks_h2"]), 3))

df = df.drop(columns="leak_clicks_h2")            # removed
print("3) honest   :", round(quick_score(FEATURES), 3), "<- the number I keep")

1) honest   : 0.568
2) with leak: 0.996
3) honest   : 0.568 <- the number I keep


**The leakage trap (measured, ROC AUC, client-holdout split).**
1. Honest score with the five features: 0.570.
2. After adding `leak_clicks_h2` (clicks from 16–31 March, the same quantity the label is built from): 0.996, close to perfect.
3. After deleting the leak column: 0.570. This is the number I keep.

The jump happened because `leak_clicks_h2` is the future outcome itself. At the decision moment (end of 15 March) it is not knowable, so a score that includes it says nothing about real performance. The honest 0.570 is only slightly above chance (0.5), so it is a directional baseline, not a finished model.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of my slice: Search Console coverage.** Only about 36.7% of March 2026 rows have `gsc_data_available IS TRUE` (measured in Q3), and the panel covers 55 clients. My slice therefore only describes content and days where Search Console data exists, and results may lean toward the clients with the most coverage.

**Other limits (observed, directional):**
- One month only, so seasonality cannot be seen.
- The label compares two 15-day halves of one month, which is a short and noisy window.
- Content with no rows in 16–31 March is treated as 0 clicks, but missing rows could mean missing data rather than zero traffic.
- The panel is unbalanced: clients start at different dates, so history depth differs.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
t = avail.loc[0, "total_rows"]
print("GSC available :", f"{avail.loc[0, 'gsc_available'] / t:.1%}")
print("GA4 available :", f"{avail.loc[0, 'ga4_available'] / t:.1%}")

GSC available : 36.7%
GA4 available : 4.2%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.